# Recomendation Engine

Technologies: XGBoost and rule Engine

## Recomendation Types

Spending Warning: User compared to themself and other uers

Savings Suggestion: Estimated monthly savings potential

Budget Advice: Top overspent category compared to others

Behavioral Nudge: Velocity, weekrnd patterns, merchant diversity

### Comparison Types
1. Personal - user compared to their own history
2. Peer - user compared to other users

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

In [6]:
fm_encoded      = pd.read_csv('../features/fm_encoded.csv', index_col='user_id')
transactions_df = pd.read_csv('../features/transactions_enriched.csv', parse_dates=['date'])
anomaly_df      = pd.read_csv('outputs/anomaly_scores.csv', index_col='user_id')

print(f'fm_encoded: {fm_encoded.shape[0]} users, {fm_encoded.shape[1]} columns')
print(f'transactions_df: {len(transactions_df):,} rows')
print(f'anomaly_df: {anomaly_df.shape[0]} users')
print(f'\ntask_segment distribution:')
print(fm_encoded['task_segment'].value_counts())

fm_encoded: 388 users, 120 columns
transactions_df: 22,602 rows
anomaly_df: 388 users

task_segment distribution:
task_segment
Single-Tasker          187
Low Activity/Trial      99
Consistent Weekly       64
Healthy Active User     33
High-Intensity User      5
Name: count, dtype: int64


## Create Peer Groups

Peer groups created from task_segment from fm_encoded. For each group calculate median value across all key features to be able compare every recomendation.

In [ ]:
PEER_BENCHMARK_COLS = [
    'vel_7d', 'vel_30d', 'avg_amt_30d', 'count_30d',
    'accounts.MEAN(transactions.amount)',
    'accounts.MAX(transactions.amount)',
    'accounts.MEAN(monthly_stats.total_spend)',
    'accounts.MAX(monthly_stats.total_spend)',
    'accounts.SUM(monthly_stats.total_spend)',
    'merchant_diversity', 'new_merchant_count',
    'days_since_last',
]

CATEGORY_COLS = [
    c for c in fm_encoded.columns
    if 'monthly_stats' in c and 'MEAN' in c and 'total' not in c and 'avg_transaction' not in c
]

available_benchmark = [c for c in PEER_BENCHMARK_COLS if c in fm_encoded.columns]
available_categories = [c for c in CATEGORY_COLS if c in fm_encoded.columns]

print(f'Benchmark features:  {len(available_benchmark)}')
print(f'Category features:   {len(available_categories)}')
print(f'{[c.split("monthly_stats.")[1].rstrip(")") for c in available_categories]}')

Benchmark features:  12
Category features:   11
['Fitness', 'Food', 'Friend Activities', 'Gifts', 'Hobbies', 'Housing and Utilities', 'Medical/Dental', 'Personal Hygiene', 'Subscriptions', 'Transportation', 'Travel']


In [9]:
peer_benchmarks = (
    fm_encoded.groupby('task_segment')[available_benchmark + available_categories].median()
)

print(f'\nPeer benchmarks computed for {len(peer_benchmarks)} segments:')
print(peer_benchmarks[available_benchmark].round(2))


Peer benchmarks computed for 5 segments:
                     vel_7d  vel_30d  avg_amt_30d  count_30d  \
task_segment                                                   
Consistent Weekly       2.0      3.0       281.26        3.0   
Healthy Active User     3.0      6.0       632.96        6.0   
High-Intensity User    22.0     66.0        -0.60       66.0   
Low Activity/Trial      0.0      0.0         0.00        0.0   
Single-Tasker           1.0      2.0       200.00        2.0   

                     accounts.MEAN(transactions.amount)  \
task_segment                                              
Consistent Weekly                               1312.37   
Healthy Active User                              692.83   
High-Intensity User                                0.01   
Low Activity/Trial                                 0.00   
Single-Tasker                                   1000.00   

                     accounts.MAX(transactions.amount)  \
task_segment                         

## XGBoost

Create a synthetic financial label using lables:
- Consistent transaction velocity 
- Average amount within a healthy range
- Not flagged as anomalous
- Positive balance

XGBoost is trained on this labels, then extract feature importance to rank witch feature most drive financial health.These weights set which rule triggers get priority in the recomendation output.

In [13]:
health_df = fm_encoded[available_benchmark].fillna(0).copy()
health_df.index = fm_encoded.index

health_df['anomaly_score'] = anomaly_df['anomaly_score'].reindex(health_df.index).fillna(50)

scores = []

for i in health_df.index:
    s = 0

    if 'vel_30d' in health_df.columns:
        v = health_df.loc[i, 'vel_30d']
        if 3 <= v <= 25:
            s += 1

    if 'avg_amt_30d' in health_df.columns:
        med = health_df['avg_amt_30d'].median()
        v = health_df.loc[i, 'avg_amt_30d']
        if 0 < v <= med * 2.5:
            s += 1

    if 'accounts.balances_current' in fm_encoded.columns:
        v = fm_encoded.loc[i, 'accounts.balances_current'] if i in fm_encoded.index else 0
        if v > 0:
            s += 1

    if 'merchant_diversity' in health_df.columns:
        med_div = health_df['merchant_diversity'].median()
        v = health_df.loc[i, 'merchant_diversity']
        if v >= med_div:
            s += 1

    if health_df.loc[i, 'anomaly_score'] < 50:
        s += 1

    scores.append(s)

score = pd.Series(scores, index=health_df.index)

health_label = []
for s in score:
    if s >= 3:
        health_label.append(1)
    else:
        health_label.append(0)

health_label = pd.Series(health_label, index=health_df.index)

print(f'Health label distribution:')
print(f'Healthy (1): {health_label.sum()} users')
print(f'At-risk (0): {(health_label == 0).sum()} users')


Health label distribution:
Healthy (1): 274 users
At-risk (0): 114 users


### Train XGBoost

In [22]:
X_xgb = health_df.copy()
y_xgb = health_label

scaler = StandardScaler()
X_xgb_scaled = scaler.fit_transform(X_xgb)

xgb = XGBClassifier(
    n_estimators=100,
    max_depth=3,          
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)
xgb.fit(X_xgb_scaled, y_xgb)

cv_scores = cross_val_score(xgb, X_xgb_scaled, y_xgb, cv=5, scoring='accuracy')
print(f'Cross-validation Accuracy: \n average: {cv_scores.mean():.3f} \n standard deviation: {cv_scores.std():.3f}')

Cross-validation Accuracy: 
 average: 0.974 
 standard deviation: 0.032


#### Extract Feature Importance

Select top 8 fitures for weight recomendations

In [23]:
importance_df = pd.DataFrame({'feature': X_xgb.columns,'importance': xgb.feature_importances_}).sort_values('importance', ascending=False)
display(importance_df)

TOP_FEATURES = importance_df.head(8)['feature'].tolist()
print(f'\nTop features: {TOP_FEATURES}')

,feature,importance
0,vel_7d,0.303634
10,new_merchant_count,0.224562
9,merchant_diversity,0.126174
1,vel_30d,0.112214
3,count_30d,0.067156
7,accounts.MAX(monthly_stats.total_spend),0.036905
12,anomaly_score,0.036476
2,avg_amt_30d,0.029847
8,accounts.SUM(monthly_stats.total_spend),0.026924
6,accounts.MEAN(monthly_stats.total_spend),0.024363



Top features: ['vel_7d', 'new_merchant_count', 'merchant_diversity', 'vel_30d', 'count_30d', 'accounts.MAX(monthly_stats.total_spend)', 'anomaly_score', 'avg_amt_30d']


## Rule Engine

Rules are ordered by XGBoost feature importance. The highest importance signal that triggers a rule becomes the primary recomendation.

In [26]:
def safe_get(row, col, default=0):
    val = row.get(col, default)
    if pd.isna(val):
        return default
    else:
        return val

##### Rule 1: Spending Warnings

Compares recent spend vs personal all-time average AND vs peer median. Returns list of warning dicts.

1. Personal: recent avg is 40%+ above all-time avg
1. Peer: spending 50%+ above peer group median
1. Unusually Large transaction count from transactions_enriched

In [27]:
def check_spending_warnings(user_row, peer_row, user_id, transactions_df):
    warnings_out = []

    avg_30d = safe_get(user_row, 'avg_amt_30d')
    all_time_avg = safe_get(user_row, 'accounts.MEAN(transactions.amount)')
    if peer_row != None:
        peer_avg = safe_get(peer_row, 'avg_amt_30d')
    else:
        peer_avg = None

    if all_time_avg > 0 and avg_30d > all_time_avg * 1.4:
        pct = round((avg_30d / all_time_avg - 1) * 100)
        warnings_out.append({
            'type': 'spending_warning',
            'comparison': 'personal',
            'priority': 'high',
            'message': f'Your average transaction amount this month (${avg_30d:.0f}) is {pct}% above your usual average (${all_time_avg:.0f}).',
            'action': 'Review your recent transactions to identify what changed.',
            'metric': {'avg_amt_30d': avg_30d, 'all_time_avg': all_time_avg, 'pct_above': pct}
        })

    if peer_avg and peer_avg > 0 and avg_30d > peer_avg * 1.5:
        pct = round((avg_30d / peer_avg - 1) * 100)
        warnings_out.append({
            'type': 'spending_warning',
            'comparison': 'peer',
            'priority': 'medium',
            'message': f'Your recent average spend (${avg_30d:.0f}/txn) is {pct}% higher than similar users (${peer_avg:.0f}/txn).',
            'action': 'Consider whether your spending aligns with your financial goals.',
            'metric': {'avg_amt_30d': avg_30d, 'peer_avg': peer_avg, 'pct_above': pct}
        })

    user_trans = transactions_df[transactions_df['user_id'] == user_id]
    if not user_trans.empty:
        large_count = (user_trans['spend_anomaly_type'] == 'Unusually Large').sum()
        if large_count >= 3:
            warnings_out.append({
                'type': 'spending_warning',
                'comparison': 'personal',
                'priority': 'high',
                'message': f'You have {large_count} unusually large transactions compared to your own average.',
                'action': 'Check these transactions — they are significantly larger than your typical spend.',
                'metric': {'large_transactions': int(large_count)}
            })

    return warnings_out

#### Rule 2: Savings Suggestions

Estimates monthly savings potential by comparing to peer median spend.

1. Peer-based: spending more than peers, potential to save
1. Low balance warning
1. Positive savings opportunity , balance is healthy but spending is high

In [28]:
def check_savings_suggestions(user_row, peer_row):
    suggestions = []

    monthly_spend = safe_get(user_row, 'accounts.MEAN(monthly_stats.total_spend)')
    if peer_row != None:
        peer_monthly_spend = safe_get(peer_row, 'accounts.MEAN(monthly_stats.total_spend)')  
    else:
        peer_monthly_spend = None
    balance = safe_get(user_row, 'accounts.balances_current')

    if peer_monthly_spend and peer_monthly_spend > 0 and monthly_spend > peer_monthly_spend * 1.2:
        potential_saving = round(monthly_spend - peer_monthly_spend, 2)
        suggestions.append({
            'type': 'savings_suggestion',
            'comparison': 'peer',
            'priority': 'medium',
            'message': f'You spend ${monthly_spend:.0f}/month on average. Similar users spend ${peer_monthly_spend:.0f}/month.',
            'action': f'Reducing to peer-level spending could save you ~${potential_saving:.0f}/month.',
            'metric': {'monthly_spend': monthly_spend, 'peer_monthly_spend': peer_monthly_spend, 'potential_saving': potential_saving}
        })

    if 0 < balance < 200:
        suggestions.append({
            'type': 'savings_suggestion',
            'comparison': 'personal',
            'priority': 'high',
            'message': f'Your current balance is low (${balance:.2f}).',
            'action': 'Consider reducing discretionary spending this month to build a buffer.',
            'metric': {'current_balance': balance}
        })

    if balance > 1000 and monthly_spend > 0:
        save_rate = round(balance / (monthly_spend + 1) * 100, 1)
        if save_rate < 50:
            suggestions.append({
                'type': 'savings_suggestion',
                'comparison': 'personal',
                'priority': 'low',
                'message': f'Your balance-to-spend ratio is {save_rate}% — you have room to save more.',
                'action': 'Setting a monthly savings target could help you grow your balance faster.',
                'metric': {'balance': balance, 'monthly_spend': monthly_spend, 'save_rate_pct': save_rate}
            })

    return suggestions

#### Rule 3: Budget Advice

Identifies top overspent category vs both personal MAX and peer median.

1. Extract user category spend
1. Top spending category for this user
1. Compare to peer median in same category
1. Find fastest-growing category

In [29]:
def check_budget_advice(user_row, peer_row, available_categories):
    advice = []
    if not available_categories:
        return advice

    user_cat_spend = {}
    for col in available_categories:
        cat_name = col.split('monthly_stats.')[1].rstrip(')')
        val = safe_get(user_row, col)
        if val > 0:
            user_cat_spend[cat_name] = val

    if not user_cat_spend:
        return advice

    top_cat = max(user_cat_spend, key=user_cat_spend.get)
    top_cat_val = user_cat_spend[top_cat]
    top_cat_col = f'accounts.MEAN(monthly_stats.{top_cat})'

    if peer_row != None and top_cat_col in peer_row.index:
        peer_cat_val = safe_get(peer_row, top_cat_col)
        if peer_cat_val > 0 and top_cat_val > peer_cat_val * 1.3:
            pct = round((top_cat_val / peer_cat_val - 1) * 100)
            advice.append({
                'type': 'budget_advice',
                'comparison': 'peer',
                'priority': 'medium',
                'message': f'You spend {pct}% more than similar users on {top_cat} (${top_cat_val:.0f} vs peer avg ${peer_cat_val:.0f}/month).',
                'action': f'Setting a monthly budget for {top_cat} could bring your spend in line with peers.',
                'metric': {'category': top_cat, 'user_spend': top_cat_val, 'peer_spend': peer_cat_val, 'pct_above': pct}
            })

    spikes = {}
    for col in available_categories:
        cat_name = col.split('monthly_stats.')[1].rstrip(')')
        mean_val = safe_get(user_row, col)
        max_col = col.replace('MEAN', 'MAX')
        if max_col in user_row.index:
            max_val = safe_get(user_row, max_col)  
        else:
            max_val = 0
        if mean_val > 0 and max_val > mean_val * 2:
            spikes[cat_name] = round(max_val / mean_val, 1)

    if spikes:
        spike_cat = max(spikes, key=spikes.get)
        spike_ratio = spikes[spike_cat]
        advice.append({
            'type': 'budget_advice',
            'comparison': 'personal',
            'priority': 'medium',
            'message': f'Your {spike_cat} spending spiked to {spike_ratio}x your usual amount in your highest month.',
            'action': f'Consider a monthly cap on {spike_cat} to avoid these irregular spikes.',
            'metric': {'category': spike_cat, 'spike_ratio': spike_ratio}
        })

    return advice

#### Rule 4: Behavioral Nudges

Surfaces patterns the user may not be aware of:
- Weekend spending concentration
- Erratic velocity
- Low merchant diversity
- New merchant exploration trend

1. Erratic velocity, burst then silence
1. Low merchant diversity vs peers
1. Weekend spending pattern from transactions
1. Inactive user nudge

In [ ]:
def check_behavioral_nudges(user_row, peer_row, user_id, transactions_df):
    nudges = []

    vel_1d = safe_get(user_row, 'vel_1d')
    vel_7d = safe_get(user_row, 'vel_7d')
    vel_30d = safe_get(user_row, 'vel_30d')
    diversity = safe_get(user_row, 'merchant_diversity')
    new_merch = safe_get(user_row, 'new_merchant_count')
    segment = user_row.get('task_segment', '')

    if peer_row is not None:
        peer_diversity = safe_get(peer_row, 'merchant_diversity')  
    else: None

    if peer_row is not None:
        peer_vel_30d = safe_get(peer_row, 'vel_30d')  
    else: None

    if vel_1d >= 4 and vel_30d > 0:
        daily_avg = vel_30d / 30
        if vel_1d > daily_avg * 5:
            nudges.append({
                'type': 'behavioral_nudge',
                'comparison': 'personal',
                'priority': 'medium',
                'message': f'You made {int(vel_1d)} transactions today — your daily average is {daily_avg:.1f}.',
                'action': 'Unusually high transaction days can indicate impulse spending. Review today\'s purchases.',
                'metric': {'vel_1d': int(vel_1d), 'daily_avg': round(daily_avg, 1)}
            })

    if peer_diversity and peer_diversity > 0 and diversity < peer_diversity * 0.5:
        nudges.append({
            'type': 'behavioral_nudge',
            'comparison': 'peer',
            'priority': 'low',
            'message': f'You shop at {int(diversity)} unique merchants — similar users visit {int(peer_diversity)} on average.',
            'action': 'You may be overly concentrated with a few merchants. Exploring alternatives could save money.',
            'metric': {'user_diversity': int(diversity), 'peer_diversity': int(peer_diversity)}
        })

    user_trans = transactions_df[transactions_df['user_id'] == user_id].copy()
    if not user_trans.empty:
        user_trans['is_weekend'] = user_trans['date'].dt.weekday >= 5
        weekend_spend = user_trans[user_trans['is_weekend']]['amount'].abs().sum()
        weekday_spend = user_trans[~user_trans['is_weekend']]['amount'].abs().sum()
        total_spend = weekend_spend + weekday_spend

        if total_spend > 0:
            weekend_pct = round(weekend_spend / total_spend * 100)  
        else: 0

        if weekend_pct > 55:
            nudges.append({
                'type': 'behavioral_nudge',
                'comparison': 'personal',
                'priority': 'low',
                'message': f'{weekend_pct}% of your total spend happens on weekends.',
                'action': 'Weekend spending patterns often include more impulse purchases. Planning ahead could reduce costs.',
                'metric': {'weekend_spend_pct': weekend_pct, 'weekend_spend': round(weekend_spend, 2), 'weekday_spend': round(weekday_spend, 2)}
            })

    days_since = safe_get(user_row, 'days_since_last')
    if days_since > 14 and vel_30d <= 2:
        nudges.append({
            'type': 'behavioral_nudge',
            'comparison': 'personal',
            'priority': 'low',
            'message': f'You have been mostly inactive — only {int(vel_30d)} transactions in the last 30 days.',
            'action': 'Check if any scheduled payments or subscriptions may have gone unnoticed.',
            'metric': {'days_since_last': int(days_since), 'vel_30d': int(vel_30d)}
        })

    return nudges